# Hand Gesture Image Classification with Neural Networks & SHAP

This notebook implements a **10-class hand-gesture image classifier** using the
LeapGestRecog dataset. It uses a multilayer perceptron (MLP) built with
TensorFlow/Keras and includes model evaluation plus SHAP-based explainability.

### Scope
- Input: grayscale images captured with a Leap Motion sensor
- Output: one of 10 gesture classes
- Model: MLP (not a live-camera system)
- Explainability: SHAP `GradientExplainer`

> This project originated as a CYBR 422 team project. The repository README
> documents individual and team contributions transparently.

## 1. Setup

The dataset is downloaded from Kaggle using `kagglehub`. The notebook does **not**
require a local Windows path or a Google Drive mount.

In [ ]:
# If needed in Google Colab:
# !pip install -q kagglehub opencv-python-headless tensorflow scikit-learn seaborn shap tqdm

import os
import glob
import random
from pathlib import Path

import cv2
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import tensorflow as tf

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Dense, Flatten, LeakyReLU
from tensorflow.keras.models import Sequential

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 2. Download and locate the LeapGestRecog dataset

In [ ]:
dataset_root = kagglehub.dataset_download("gti-upm/leapgestrecog")
print("Dataset root:", dataset_root)

matches = glob.glob(os.path.join(dataset_root, "**", "leapGestRecog"), recursive=True)
if not matches:
    raise FileNotFoundError("Could not locate the leapGestRecog directory.")

data_dir = Path(matches[0])
print("Using:", data_dir)

## 3. Define the 10 gesture classes

A key part of the project was ensuring that each distinct gesture maps to its own
class ID rather than being grouped incorrectly.

In [ ]:
CLASS_NAMES = [
    "01_palm",
    "02_l",
    "03_fist",
    "04_fist_moved",
    "05_thumb",
    "06_index",
    "07_ok",
    "08_palm_moved",
    "09_c",
    "10_down",
]

class_to_id = {name: idx for idx, name in enumerate(CLASS_NAMES)}
for name, idx in class_to_id.items():
    print(f"Class {idx}: {name}")

## 4. Build the sampled dataset

The recorded course-notebook run used **5 images per gesture per subject**.
With 10 subjects and 10 gestures, this produced **500 images**.

The images are loaded in grayscale and resized to `240 × 640`.

In [ ]:
IMG_WIDTH = 640
IMG_HEIGHT = 240
IMAGES_PER_CLASS_PER_SUBJECT = 5

def load_dataset(data_dir, images_per_class=5):
    samples = []

    subject_dirs = sorted(
        p for p in data_dir.iterdir()
        if p.is_dir() and p.name.isdigit()
    )

    for subject_dir in subject_dirs:
        for class_name in CLASS_NAMES:
            class_dir = subject_dir / class_name
            if not class_dir.exists():
                continue

            image_files = sorted(
                p for p in class_dir.iterdir()
                if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp"}
            )[:images_per_class]

            class_id = class_to_id[class_name]

            for image_path in image_files:
                image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
                if image is None:
                    continue
                image = cv2.resize(image, (IMG_WIDTH, IMG_HEIGHT))
                samples.append((image, class_id))

    return samples

samples = load_dataset(data_dir, IMAGES_PER_CLASS_PER_SUBJECT)
print("Total samples:", len(samples))

## 5. Preprocess and split the data

In [ ]:
random.shuffle(samples)

X = np.array([image for image, _ in samples], dtype=np.float32) / 255.0
y = np.array([label for _, label in samples], dtype=np.int32)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

## 6. Build the MLP classifier

The project uses a dense neural network as a proof of concept. Because the input
images are flattened, this model does not preserve spatial structure the way a CNN
would. A CNN is listed as a future improvement in the repository README.

In [ ]:
model = Sequential([
    Flatten(input_shape=(IMG_HEIGHT, IMG_WIDTH)),
    Dense(64),
    LeakyReLU(negative_slope=0.1),
    Dense(32),
    LeakyReLU(negative_slope=0.1),
    Dense(16),
    LeakyReLU(negative_slope=0.1),
    Dense(len(CLASS_NAMES), activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## 7. Train the model

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    validation_split=0.10,
    batch_size=32,
    verbose=2,
)

## 8. Evaluate performance

In [ ]:
train_loss, train_accuracy = model.evaluate(X_train, y_train, verbose=0)
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"Training loss: {train_loss:.4f}")
print(f"Training accuracy: {train_accuracy:.2%}")
print(f"Testing loss: {test_loss:.4f}")
print(f"Testing accuracy: {test_accuracy:.2%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="validation")
axes[0].set_title("Model Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="validation")
axes[1].set_title("Model Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Predictions, confusion matrix, and classification report

In [ ]:
probabilities = model.predict(X_test)
y_pred = np.argmax(probabilities, axis=1)

print(classification_report(
    y_test,
    y_pred,
    target_names=CLASS_NAMES,
    zero_division=0,
))

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(11, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
sample_idx = 5

prediction = model.predict(X_test[sample_idx:sample_idx + 1], verbose=0)[0]
predicted_class = int(np.argmax(prediction))
confidence = float(np.max(prediction))

plt.imshow(X_test[sample_idx], cmap="gray")
plt.title(
    f"Actual: {CLASS_NAMES[y_test[sample_idx]]}\n"
    f"Predicted: {CLASS_NAMES[predicted_class]} ({confidence:.2%})"
)
plt.axis("off")
plt.show()

## 10. Explainability with SHAP

`GradientExplainer` is used to estimate which image regions contributed to the
model's output. SHAP is an explainability tool; it does not by itself prove that a
model is unbiased or trustworthy.

In [ ]:
BACKGROUND_SIZE = min(50, len(X_train))
EXPLAIN_SIZE = min(10, len(X_test))

background = X_train[:BACKGROUND_SIZE]
explain_batch = X_test[:EXPLAIN_SIZE]

explainer = shap.GradientExplainer(model, background)
shap_values = explainer.shap_values(explain_batch)

In [ ]:
sample_idx = 5

if sample_idx >= EXPLAIN_SIZE:
    raise IndexError("sample_idx must be within the SHAP explanation batch.")

prediction = model.predict(
    explain_batch[sample_idx:sample_idx + 1],
    verbose=0,
)[0]

predicted_class = int(np.argmax(prediction))
confidence = float(np.max(prediction))

print("Predicted:", CLASS_NAMES[predicted_class])
print(f"Confidence: {confidence:.2%}")
print("Actual:", CLASS_NAMES[y_test[sample_idx]])

shap.image_plot(
    shap_values,
    explain_batch[sample_idx:sample_idx + 1],
    show=False,
)
plt.gcf().set_size_inches(12, 8)
plt.show()

## 11. Limitations and next steps

- The recorded run uses a **small 500-image subset** of the full dataset.
- The MLP flattens the image and therefore does not exploit spatial structure.
- A **CNN** would be a more appropriate architecture for image classification.
- Validation should be expanded and repeated across multiple random seeds.
- A subject-aware split would better test generalization to unseen people.
- SHAP visualizations should be interpreted as model explanations, not guarantees of fairness.
- A Leap Motion sensor would be required for a true real-time sensor pipeline; this project classifies stored images.